# ENCODE ChIP-seq QC validation of EpiClass Input-class probability

## Purpose

Reviewer response analysis: do EpiClass low-confidence / high Input-class predictions correspond to ChIP-seq experiments that ENCODE flags as poor quality by conventional metrics? Provides external quantitative validation of the manuscript's interpretation that low EpiClass prediction scores and elevated Input-class probabilities reflect noisy ChIP datasets.

## Inputs

1. **ENCODE quality metric objects** (JSON), one list of metric entries per experiment accession, fetched from the ENCODE portal. Multiple metric types per experiment, computed at different pipeline stages.
2. **EpiClass predictions** on ENCODE ChIP-seq files, with per-class softmax probabilities including the Input class.

## Pipeline

### 1. Restrict QC metrics

Restrict ENCODE QC metrics to ChIP-seq experiments present in the EpiClass prediction table.

### 2. Build per-experiment QC table

- Extract a curated set of fields from each metric type (FRiP, JSD, NSC, RSC, NRF, PBC1/2, peak counts, library complexity, depth).
- Aggregate across replicate-level entries (mean, with min/max/n).
- Coalesce metrics that appear in multiple metric types (e.g. NRF in both `ChipLibraryQualityMetric` and the older `ChipSeqFilterQualityMetric`) into single consensus columns, preferring the more populated source.
- Tag pipeline version (new vs legacy) from which metric types are present; drop the small legacy subset to avoid batch effects from non-comparable NSC/RSC computations across pipeline versions.

### 3. Filter EpiClass predictions

- **Exclude Input control experiments.** The reviewer's hypothesis is about ChIP samples, not controls; including Inputs would inflate correlations trivially since real Inputs both have low FRiP/NSC and high Input-class probability.
- **Keep only experiments with a single final bigwig file**, so that the QC metrics unambiguously correspond to the file EpiClass scored.
- **Tag each experiment as "core" (histone marks) or "non-core"** (TFs and chromatin regulators), since these go through different ENCODE pipelines with different reproducibility logic.

### 4. Join

Join QC table to filtered predictions on `EXPERIMENT_accession`.

### 5. Spearman correlations

Compute Spearman correlations between `log10(Input-class probability)` and each of ten QC metrics, separately for histones and non-histones:

- **Pooled** correlation across all experiments in the group.
- **Within-target** correlations (per histone mark or per TF target), restricted to targets with ≥20 experiments, then aggregated by Fisher z-transformation weighted by `(n - 3)`. Controls for Simpson's paradox driven by between-target differences in baseline QC distributions (e.g. broad vs narrow histone marks differ in typical FRiP by an order of magnitude).

> **Note:** max-prediction-score correlations were tried first but are unusable for histones due to ceiling effects (>90% of experiments score >0.99). Input-class probability avoids this saturation because it has meaningful variance even for confidently-predicted samples.

### 6. Split histones into narrow vs broad

Split histones into:

- **Narrow:** H3K4me3, H3K27ac, H3K4me1
- **Broad:** H3K27me3, H3K36me3, H3K9me3

Re-run the correlation analysis on each group. This separation is biologically motivated: NSC, RSC, and to a lesser extent FRiP and peak counts are known to be insensitive for broad chromatin marks (Landt et al. 2012; Nakato & Sakata 2021), so the two groups are expected to behave differently as comparison standards.

### 7. Plot

Three-panel bar chart: TFs, narrow histones, broad histones. Pooled correlations in grey, within-target/within-mark meta correlations in color. Negative ρ indicates the reviewer-predicted direction.

## Key findings

- **TFs:** clean negative correlations on all five signal-to-noise metrics in both pooled and within-target meta (within-target ρ between −0.15 and −0.38). Directly supports the reviewer's interpretation.
- **Narrow histones:** pooled correlations are strong and negative but collapse to ~0 within-mark, reflecting both Simpson's paradox and the high confidence of EpiClass on ENCODE histones (only 13 of 3,549 experiments below the operational threshold of 0.6).
- **Broad histones:** neither pooled nor within-mark correlations are negative, consistent with the documented insensitivity of conventional QC metrics for broad chromatin marks.

## Output

Figure exported as `encode_qc_correlations_3panels.png`. Intended as an extended figure for the manuscript (TF panel) and full reviewer figure in the response letter (all three panels).

## SETUP

In [ ]:
# pylint: disable=import-error, redefined-outer-name, too-many-lines, use-dict-literal, missing-module-docstring
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

In [ ]:
base_dir = Path.home() / "Projects/epiclass/output/paper"
metadata_dir = base_dir / "data/metadata"

encode_metadata_dir = metadata_dir / "encode"

preds_dir = base_dir / "data" / "training_results"

for path in [metadata_dir, encode_metadata_dir, preds_dir]:
    if not path.exists():
        raise ValueError(f"Path {path} does not exist.")

In [ ]:
qc_metrics_path = (
    encode_metadata_dir
    / "encode_experiment_quality_metrics_2025-02_no_revoked_hg38.freeze1.json"
)
with open(qc_metrics_path, "r", encoding="utf8") as f:
    qc_metrics = json.load(f)
print(f"QC metrics N: {len(qc_metrics)}")

In [ ]:
filepath = (
    preds_dir
    / "dfreeze_v2/predictions/encode/complete_encode_predictions_augmented_2025-02_metadata.csv.gz"
)
encode_preds = pd.read_csv(filepath, low_memory=False)
display(encode_preds.shape)

## Acquire ChIP QC metrics

In [ ]:
preds_chip = encode_preds[
    encode_preds["FILE_assay_title"].str.contains("chip", case=False, na=False)
]
print(f"ChIP-seq metadata shape: {preds_chip.shape}")

In [ ]:
exp_id_label = "EXPERIMENT_accession"
chip_exp_acc = preds_chip[exp_id_label].unique()
print(f"Number of unique ChIP-seq experiments: {len(chip_exp_acc)}")

qc_metrics = {k: v for k, v in qc_metrics.items() if k in chip_exp_acc}
print(f"QC metrics N after filtering: {len(qc_metrics)}")

In [ ]:
# Sanity check: Ensure that all experiments in qc_metrics have the 'quality_metric' key
for acc, metrics_list in qc_metrics.items():
    for metrics in metrics_list:
        if "quality_metric" not in metrics:
            print(f"Missing 'quality_metric' key for experiment {acc}")

In [ ]:
metrics_type_count = Counter()
metrics_type_entries = defaultdict(set)
for acc in qc_metrics:
    all_metrics = qc_metrics[acc]
    # print(f"Experiment {acc} has {len(all_metrics)} quality metrics entries.")
    for metrics in all_metrics:
        details = metrics["quality_metric"]

        # available
        metrics_type_count.update([details["@type"][0]])

        # track which keys are present for each metric type
        metrics_type_entries[details["@type"][0]].update(details.keys())

In [ ]:
display(metrics_type_count.most_common())

display(metrics_type_entries)

In [ ]:
FIELDS_BY_TYPE = {
    "ChipPeakEnrichmentQualityMetric": ["frip"],
    "ChipReplicationQualityMetric": ["reproducible_peaks"],
    "ChipAlignmentEnrichmentQualityMetric": [
        "jsd",
        "NSC",
        "RSC",
        "pct_genome_enrich",
        "diff_enrich",
    ],
    "ChipLibraryQualityMetric": ["NRF", "PBC1", "PBC2"],
    "ChipSeqFilterQualityMetric": ["NRF", "PBC1", "PBC2", "NSC", "RSC"],
    "ChipAlignmentQualityMetric": [
        "usable_fragments",
        "mapped_reads",
        "pct_mapped_reads",
    ],
    "HistoneChipSeqQualityMetric": ["frip", "npeak_overlap", "nreads_in_peaks"],
}

In [ ]:
def collect_experiment_metrics(qc_metrics):
    """
    Build a per-experiment QC table.

    Parameters
    ----------
    qc_metrics : dict[str, list[dict]]
        Mapping experiment accession -> list of metric entries, where each
        entry has a 'quality_metric' dict containing '@type' and metric fields.

    Returns
    -------
    pd.DataFrame
        One row per experiment. For each (metric_type, field) pair, columns:
            <type>__<field>__mean
            <type>__<field>__n      (number of files contributing)
        Plus *_min / *_max for fields where spread matters.
    """
    # raw[acc][(type, field)] = list of values across files
    raw = defaultdict(lambda: defaultdict(list))

    for acc, entries in qc_metrics.items():
        for entry in entries:
            details = entry["quality_metric"]
            mtype = details["@type"][0]
            if mtype not in FIELDS_BY_TYPE:
                continue
            for field in FIELDS_BY_TYPE[mtype]:
                val = details.get(field)
                if val is None:
                    continue
                try:
                    raw[acc][(mtype, field)].append(float(val))
                except (TypeError, ValueError):
                    continue

    rows = []
    for acc, metric_dict in raw.items():
        row = {"experiment": acc}
        for (mtype, field), values in metric_dict.items():
            arr = np.asarray(values, dtype=float)
            prefix = f"{mtype}__{field}"
            row[f"{prefix}__mean"] = float(np.nanmean(arr))
            row[f"{prefix}__n"] = int(np.sum(~np.isnan(arr)))
            if arr.size > 1:
                row[f"{prefix}__min"] = float(np.nanmin(arr))
                row[f"{prefix}__max"] = float(np.nanmax(arr))
        rows.append(row)

    df = pd.DataFrame(rows).set_index("experiment")
    return df

In [ ]:
def add_consensus_columns(df):
    """
    Add consensus QC metric columns by coalescing across metric type sources.

    Several ENCODE QC metrics appear under multiple metric types because the
    ENCODE pipeline has been versioned over time and because the histone and
    TF pipelines emit different metric objects. For example, NRF/PBC1/PBC2/
    NSC/RSC are present in the legacy `ChipSeqFilterQualityMetric` and were
    later split into `ChipLibraryQualityMetric` (complexity) and
    `ChipAlignmentEnrichmentQualityMetric` (cross-correlation) in the newer
    pipeline; FRiP appears in `ChipPeakEnrichmentQualityMetric` (uniform
    source) as well as in the pipeline-specific `HistoneChipSeqQualityMetric`
    and `IDRQualityMetric` summaries.

    This function creates a single short-named column per metric (e.g. `frip`,
    `NSC`, `NRF`) by coalescing across the relevant source columns produced
    by `collect_experiment_metrics`, preferring the more uniformly populated
    source and falling back to pipeline-specific sources as needed.

    Peak counts are deliberately NOT coalesced into a single cross-pipeline
    column because the histone pipeline (`npeak_overlap`, overlap-based) and
    the TF pipeline (`N_optimal`, IDR-based) define them differently. Two
    separate columns `peaks_histone` and `peaks_tf` are created instead, plus
    a `peaks_any` convenience column intended only for within-class analyses.

    Parameters
    ----------
    df : pd.DataFrame
        Per-experiment QC table from `collect_experiment_metrics`, indexed by
        experiment accession, with columns named `<metric_type>__<field>__mean`.

    Returns
    -------
    pd.DataFrame
        The input DataFrame with the following consensus columns added:
        `frip`, `peaks_histone`, `peaks_tf`, `peaks_any`, `jsd`, `NSC`, `RSC`,
        `NRF`, `PBC1`, `PBC2`, `pct_genome_enrich`, `reproducible_peaks`,
        `usable_fragments`. Any consensus column whose source columns are all
        absent will be filled with NaN.
    """

    def coalesce(*cols):
        """
        Combine multiple columns into a single Series, preferring earlier
        columns and filling missing values from later ones.

        For each row, takes the value from the first column in `cols` that
        exists in `df` and is non-null; if that value is null, falls through
        to the next column, and so on. Columns listed in `cols` that do not
        exist in `df` are silently skipped, allowing the function to be
        called with a superset of possible source columns. If none of the
        listed columns exist in `df`, returns an all-NaN Series with the
        same index.

        Parameters
        ----------
        *cols : str
            Column names to coalesce, in order of preference (most preferred
            first).

        Returns
        -------
        pd.Series
            Coalesced values aligned to `df.index`.
        """
        existing = [c for c in cols if c in df.columns]
        if not existing:
            return pd.Series(np.nan, index=df.index)
        out = df[existing[0]].copy()
        for c in existing[1:]:
            out = out.fillna(df[c])
        return out

    # FRiP: prefer the uniform source, fall back to pipeline-specific
    df["frip"] = coalesce(
        "ChipPeakEnrichmentQualityMetric__frip__mean",
        "HistoneChipSeqQualityMetric__frip__mean",
        "IDRQualityMetric__frip__mean",
    )

    # Peak count: keep histone and TF separate
    df["peaks_histone"] = coalesce("HistoneChipSeqQualityMetric__npeak_overlap__mean")
    df["peaks_tf"] = coalesce("IDRQualityMetric__N_optimal__mean")
    # Optional unified column for within-class analyses only — DO NOT pool across classes
    df["peaks_any"] = df["peaks_histone"].fillna(df["peaks_tf"])

    # Cross-pipeline-safe metrics
    df["jsd"] = coalesce("ChipAlignmentEnrichmentQualityMetric__jsd__mean")
    df["NSC"] = coalesce(
        "ChipAlignmentEnrichmentQualityMetric__NSC__mean",
        "ChipSeqFilterQualityMetric__NSC__mean",
    )
    df["RSC"] = coalesce(
        "ChipAlignmentEnrichmentQualityMetric__RSC__mean",
        "ChipSeqFilterQualityMetric__RSC__mean",
    )
    df["NRF"] = coalesce(
        "ChipLibraryQualityMetric__NRF__mean",
        "ChipSeqFilterQualityMetric__NRF__mean",
    )
    df["PBC1"] = coalesce(
        "ChipLibraryQualityMetric__PBC1__mean",
        "ChipSeqFilterQualityMetric__PBC1__mean",
    )
    df["PBC2"] = coalesce(
        "ChipLibraryQualityMetric__PBC2__mean",
        "ChipSeqFilterQualityMetric__PBC2__mean",
    )
    df["pct_genome_enrich"] = coalesce(
        "ChipAlignmentEnrichmentQualityMetric__pct_genome_enrich__mean"
    )
    df["reproducible_peaks"] = coalesce(
        "ChipReplicationQualityMetric__reproducible_peaks__mean"
    )
    df["usable_fragments"] = coalesce(
        "ChipAlignmentQualityMetric__usable_fragments__mean"
    )
    return df

In [ ]:
df_full = collect_experiment_metrics(qc_metrics)
df_full = add_consensus_columns(df_full)

`ChipSeqFilterQualityMetric` vs `ChipLibraryQualityMetric` + `ChipAlignmentEnrichmentQualityMetric`

These are from different pipeline versions. ChipSeqFilterQualityMetric is the older combined metric (NRF, PBC1, PBC2, NSC, RSC all in one object) from the legacy pipeline; the newer pipeline split these into `ChipLibraryQualityMetric` (complexity) and `ChipAlignmentEnrichmentQualityMetric` (cross-correlation). Your current coalesce handles this correctly by falling back, but be aware that mixing experiments processed by different pipeline versions introduces a batch effect. The newer pipeline's NSC/RSC values aren't perfectly comparable to the older one's because the underlying tools and subsampling differ. For the reviewer response I'd add a column tracking which pipeline version each experiment came from (you can infer it from which metric type is present) and at minimum check that the correlation doesn't flip sign between the two subsets.

In [ ]:
# After building df_full, tag pipeline version:
df_full["pipeline_version"] = np.where(
    df_full["ChipLibraryQualityMetric__NRF__mean"].notna(),
    "new",
    np.where(
        df_full["ChipSeqFilterQualityMetric__NRF__mean"].notna(), "legacy", "unknown"
    ),
)

In [ ]:
df_full["pipeline_version"].value_counts(dropna=False)

Only 10 legacy pipeline datasets, excluding them

In [ ]:
df_full = df_full[df_full["pipeline_version"] == "new"]
display(df_full.shape)

In [ ]:
# Compact table
key_cols = [
    "frip",
    "reproducible_peaks",
    "jsd",
    "NSC",
    "RSC",
    "NRF",
    "PBC1",
    "PBC2",
    "pct_genome_enrich",
    "usable_fragments",
]
df_key = df_full[key_cols]

print(f"Experiments with data: {len(df_key)}")
print(f"Coverage per metric:\n{df_key.notna().sum()}")

We're focusing on the effect of the input prediction score for non-control experiments, so we exclude input files.

In [ ]:
# excluding input
encode_preds = encode_preds[
    encode_preds["assay_epiclass"].str.contains("h3*|core", case=False, regex=True)
]
print(f"ChIP-seq files after input filtering: {encode_preds.shape[0]}")

We also exclude experiments with more than 1 file, since we can't be sure which file the QC metrics correspond to.

In [ ]:
groupby_experiment = encode_preds.groupby("EXPERIMENT_accession")
files_per_exp = groupby_experiment["FILE_accession"].nunique()

single_file_exps = files_per_exp[files_per_exp == 1].index
print(f"Experiments with single file: {len(single_file_exps)}")

In [ ]:
single_file_preds = encode_preds[
    encode_preds["EXPERIMENT_accession"].isin(single_file_exps)
]
print(f"Predictions for single-file experiments: {single_file_preds.shape}")

Analysis will treat core/non-core separately, so we tag them

In [ ]:
# h3* are core, the rest are non-core
single_file_preds.loc[:, "experiment_group"] = single_file_preds.loc[
    :, "assay_epiclass"
].apply(lambda x: "core" if "h3" in x.lower() else "non-core")

Acquire all unique assay families from ENCODE non-core
~ish, reduces integer variations to one name

In [ ]:
def strip_trailing_int(s):
    """Strip trailing digits (optionally preceded by - or _ or whitespace) from a string."""
    new = re.sub(r"[-_\s]*\d+$", "", s)
    return new + "*" if new != s else s


families = sorted({strip_trailing_int(a) for a in single_file_preds["assay"].unique()})


# for group in families:
#     print(group)

## Spearman correlation analysis

In [ ]:
INPUT_COL = "input (assay_epiclass_11c)"
SCORE_COL = "Max pred (assay_epiclass_11c)"

In [ ]:
for assay in single_file_preds["assay_epiclass"].unique():
    sub_df = single_file_preds[single_file_preds["assay_epiclass"] == assay]
    pred_scores = sub_df["Max pred (assay_epiclass_11c)"]
    print(assay)
    # print(sub_df["Max pred (assay_epiclass_11c)"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
    # print(f"\nFraction >0.99: {(sub_df['Max pred (assay_epiclass_11c)'] > 0.99).mean():.3f}")
    # print(f"Fraction >0.95: {(sub_df['Max pred (assay_epiclass_11c)'] > 0.95).mean():.3f}")

    print(
        sub_df["input (assay_epiclass_11c)"].describe(
            percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
        )
    )
    print()

In [ ]:
# Log-transform Input probability (it spans orders of magnitude)
# Add a small floor to avoid log(0)
single_file_preds.loc[:, "log_input"] = np.log10(
    single_file_preds.loc[:, INPUT_COL].clip(lower=1e-10)
)

In [ ]:
merged = pd.DataFrame.join(
    single_file_preds.set_index("EXPERIMENT_accession"),
    df_key,
    how="inner",
)
print(f"Merged table shape: {merged.shape}")

core = merged[merged["experiment_group"] == "core"].copy()
print(f"Core experiments: {len(core)}")
non_core = merged[merged["experiment_group"] == "non-core"].copy()
print(f"Non-core experiments: {len(non_core)}")

In [ ]:
core_correct = core[
    core["True class (assay_epiclass_11c)"]
    == core["Predicted class (assay_epiclass_11c)"]
]

In [ ]:
for df in core, core_correct:
    print(f"Shape: {df.shape}")
    print(f"Fraction with FRiP > 0.6: {(df['frip'] > 0.6).mean():.3f}")
    for mark, mark_df in df.groupby("assay"):
        n_low = (mark_df[SCORE_COL] < 0.6).sum()
        print(f"{mark:12s}  n_total={len(mark_df):4d}  n_below_0.6={n_low}")
    print()

In [ ]:
# Check how many low-confidence predictions (score < 0.6) we have, and their QC profiles
LOW_CONF_THRESHOLD = 0.6

low = core_correct[core_correct[SCORE_COL] < LOW_CONF_THRESHOLD]

print(f"Low-confidence: n={len(low)}")
display(
    low[
        ["assay", SCORE_COL, "frip", "reproducible_peaks", "jsd", "NSC", "NRF"]
    ].sort_values(SCORE_COL)
)

In [ ]:
print("Input probability distribution:")
for name, df in [("core", core), ("non-core", non_core)]:
    print(f"\n{name} (n={len(df)}):")
    print(df[INPUT_COL].describe())
    print(f"  log10 range: [{df['log_input'].min():.2f}, {df['log_input'].max():.2f}]")

In [ ]:
def correlate_input_with_qc(
    merged,
    qc_cols,
    target_col="assay",
    score_col="log_input",
    min_n=20,
):
    """
    Spearman correlations between Input-class probability and QC metrics,
    pooled and stratified by target.
    """
    # Pooled within group
    pooled = []
    for col in qc_cols:
        sub = merged[[score_col, col]].dropna()
        if len(sub) < min_n:
            continue
        rho, p = spearmanr(sub[score_col], sub[col])
        pooled.append({"metric": col, "spearman_rho": rho, "p_value": p, "n": len(sub)})
    pooled = pd.DataFrame(pooled)

    # Within target
    within = []
    for target, tg in merged.groupby(target_col):
        if len(tg) < min_n:
            continue
        for col in qc_cols:
            sub = tg[[score_col, col]].dropna()
            if len(sub) < min_n:
                continue
            rho, p = spearmanr(sub[score_col], sub[col])
            within.append(
                {
                    "target": target,
                    "metric": col,
                    "spearman_rho": rho,
                    "p_value": p,
                    "n": len(sub),
                }
            )
    within = pd.DataFrame(within)

    # Fisher-z meta-analysis
    if len(within):

        def fisher_mean(g):
            """Compute the weighted mean of Spearman rhos using Fisher z-transform."""
            z = np.arctanh(g["spearman_rho"].clip(-0.999, 0.999))
            w = g["n"] - 3
            z_bar = np.average(z, weights=w)
            return pd.Series(
                {
                    "spearman_rho_meta": np.tanh(z_bar),
                    "n_targets": len(g),
                    "n_total": int(g["n"].sum()),
                }
            )

        meta = (
            within.groupby("metric", group_keys=False)  # type: ignore
            .apply(fisher_mean, include_groups=False)  # type: ignore
            .reset_index()
        )
    else:
        meta = pd.DataFrame()

    return pooled, within, meta

In [ ]:
qc_cols = [
    "frip",
    "reproducible_peaks",
    "jsd",
    "NSC",
    "RSC",
    "NRF",
    "PBC1",
    "PBC2",
    "pct_genome_enrich",
    "usable_fragments",
]

print("\n\n=== CORE (histones) ===")
core_pooled, core_within, core_meta = correlate_input_with_qc(core, qc_cols)
print("\nPooled:")
print(core_pooled.round(3).to_string(index=False))
print("\nMeta-analyzed within-mark:")
print(core_meta.round(3).to_string(index=False))

print("\n\n=== NON-CORE (TFs) ===")
nc_pooled, nc_within, nc_meta = correlate_input_with_qc(non_core, qc_cols)
print("\nPooled:")
print(nc_pooled.round(3).to_string(index=False))
print("\nMeta-analyzed within-target:")
print(nc_meta.round(3).to_string(index=False))

In [ ]:
def classify_mark(target):
    """Classify a histone mark as narrow or broad based on its name."""
    narrow_marks = {"h3k4me3", "h3k27ac", "h3k4me1"}
    broad_marks = {"h3k27me3", "h3k36me3", "h3k9me3"}

    t = target.lower()
    if t in narrow_marks:
        return "narrow"
    if t in broad_marks:
        return "broad"
    return None


# Within `core`, tag each row
core["mark_class"] = core["assay"].apply(classify_mark)

# Split and run the existing correlate_input_with_qc separately
core_narrow = core[core["mark_class"] == "narrow"]
core_broad = core[core["mark_class"] == "broad"]

print(f"Narrow marks: n={len(core_narrow)} across {core_narrow['assay'].nunique()} marks")
print(f"Broad marks:  n={len(core_broad)} across {core_broad['assay'].nunique()} marks")

narrow_pooled, narrow_within, narrow_meta = correlate_input_with_qc(core_narrow, qc_cols)
broad_pooled, broad_within, broad_meta = correlate_input_with_qc(core_broad, qc_cols)

print("\n=== NARROW HISTONES ===")
print("\nWithin-mark meta:")
print(narrow_meta.round(3).to_string(index=False))

print("\n=== BROAD HISTONES ===")
print("\nWithin-mark meta:")
print(broad_meta.round(3).to_string(index=False))

### Graph

In [ ]:
def collect_for_plot(pooled_df, meta_df, metrics):
    """Helper to align pooled and meta-analyzed results for plotting."""
    pooled = pooled_df.set_index("metric")["spearman_rho"].to_dict()
    meta = meta_df.set_index("metric")["spearman_rho_meta"].to_dict()
    n_meta = meta_df.set_index("metric")["n_total"].to_dict()
    return (
        [pooled.get(m, float("nan")) for m in metrics],
        [meta.get(m, float("nan")) for m in metrics],
        [int(n_meta.get(m, 0)) for m in metrics],
    )


PLOT_METRICS = ["frip", "reproducible_peaks", "jsd", "NSC", "RSC"]
METRIC_LABELS = {
    "frip": "FRiP",
    "reproducible_peaks": "Repro.<br>peaks",
    "jsd": "JSD",
    "NSC": "NSC",
    "RSC": "RSC",
}
x_labels = [METRIC_LABELS[m] for m in PLOT_METRICS]

tf_p, tf_m, tf_n = collect_for_plot(nc_pooled, nc_meta, PLOT_METRICS)
nar_p, nar_m, nar_n = collect_for_plot(narrow_pooled, narrow_meta, PLOT_METRICS)
bro_p, bro_m, bro_n = collect_for_plot(broad_pooled, broad_meta, PLOT_METRICS)

In [ ]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        f"<b>A.</b> Transcription factors<br><span style='font-size:14px'>n={sum(tf_n)//len(tf_n)}, 17 targets</span>",
        f"<b>B.</b> Narrow histone marks<br><span style='font-size:14px'>n={len(core_narrow)}, {core_narrow['assay'].nunique()} marks</span>",
        f"<b>C.</b> Broad histone marks<br><span style='font-size:14px'>n={len(core_broad)}, {core_broad['assay'].nunique()} marks</span>",
    ),
    horizontal_spacing=0.08,
    shared_yaxes=True,
)

POOLED_COLOR = "rgba(150, 150, 150, 0.55)"
TF_COLOR = "rgba(31, 119, 180, 0.95)"
NARROW_COLOR = "rgba(44, 160, 44, 0.95)"
BROAD_COLOR = "rgba(214, 39, 40, 0.95)"

panels = [
    (1, tf_p, tf_m, tf_n, TF_COLOR, "Within-target meta"),
    (2, nar_p, nar_m, nar_n, NARROW_COLOR, "Within-mark meta (narrow)"),
    (3, bro_p, bro_m, bro_n, BROAD_COLOR, "Within-mark meta (broad)"),
]

for col, pooled, meta, n_vals, color, meta_label in panels:
    show_single_count = col in (2, 3)
    label_idx = 4  # only one bar has different n

    meta_text = (
        [f"n={n}" if i == label_idx else "" for i, n in enumerate(n_vals)]
        if show_single_count
        else None
    )

    fig.add_trace(
        go.Bar(
            x=x_labels,
            y=pooled,
            name="Pooled" if col == 1 else None,
            marker_color=POOLED_COLOR,
            offsetgroup="pooled",
            showlegend=(col == 1),
            hovertemplate="%{x}<br>Pooled ρ = %{y:.3f}<extra></extra>",
        ),
        row=1,
        col=col,
    )

    fig.add_trace(
        go.Bar(
            x=x_labels,
            y=meta,
            name=meta_label,
            marker_color=color,
            offsetgroup="meta",
            text=meta_text,
            textposition="outside" if show_single_count else None,
            textfont=dict(size=12),
            cliponaxis=False,
            constraintext="none",
            showlegend=True,
            customdata=n_vals,
            hovertemplate="%{x}<br>Meta ρ = %{y:.3f}<br>n=%{customdata}<extra></extra>",
        ),
        row=1,
        col=col,
    )

    fig.add_hline(y=0, line_width=1, line_color="black", row=1, col=col)  # type: ignore

# Y axis
all_vals = tf_p + tf_m + nar_p + nar_m + bro_p + bro_m
y_min = min(all_vals + [-0.7]) - 0.05
y_max = max(all_vals + [0.1]) + 0.15

for col in (1, 2, 3):
    fig.update_yaxes(
        range=[y_min, y_max],
        zeroline=False,
        gridcolor="rgba(0,0,0,0.08)",
        row=1,
        col=col,
    )
    fig.update_xaxes(tickangle=-30, row=1, col=col)

fig.update_yaxes(
    title_text="Spearman ρ (Input prob. vs QC metric)",
    row=1,
    col=1,
)

fig.update_layout(
    title=dict(
        text="EpiClass Input-class probability vs ENCODE ChIP-seq QC metrics",
        x=0.5,
        xanchor="center",
    ),
    barmode="group",
    bargap=0.25,
    bargroupgap=0.08,
    plot_bgcolor="white",
    width=1400,
    height=580,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="center",
        x=0.5,
        font=dict(size=15),
    ),
    margin=dict(t=110, b=130, l=70, r=30),
)

fig.show()

fig.write_image(Path.home() / "downloads" / "encode_qc_correlations_3panels.png", scale=2)